In [1]:
from bs4 import BeautifulSoup
import json
import pandas as pd
import re
import requests


In [2]:
root = "https://immovlan.be/en"
end_point = "real-estate"
params_first_page = "transactiontypes=for-sale&propertytypes=house,apartment&isnewconstruction=no&islifeannuity=no&noindex=1"
try:
    response = requests.get(f"{root}/{end_point}?{params_first_page}", headers={"User-Agent": "max-exercice"})
    
except BaseException as ex:
    print(0, ex)
else:
    print(1, response.status_code)



1 200


In [3]:
soup = BeautifulSoup(response.content, "html")

titles = soup.find_all("h2", attrs={"class": "card-title"})
all_titles = soup.find_all("h2")
print(titles == all_titles)


True


In [4]:
link_titles = soup.select('h2 a')
print(f"Number of links per page: {len(link_titles)}")
for l in link_titles:
    print(l.get("href"))

Number of links per page: 20
https://immovlan.be/en/detail/residence/for-sale/1150/sint-pieters-woluwe/vbe35189
https://immovlan.be/en/detail/residence/for-sale/6040/jumet/vbe35169
https://immovlan.be/en/detail/residence/for-sale/6183/trazegnies/vbe35167
https://immovlan.be/en/detail/residence/for-sale/5060/moignelee/vbe35133
https://immovlan.be/en/detail/residence/for-sale/5640/mettet/vbe35132
https://immovlan.be/en/detail/duplex/for-sale/5000/namur/vbe35115
https://immovlan.be/en/detail/apartment/for-sale/2390/oostmalle/rbw20430
https://immovlan.be/en/detail/penthouse/for-sale/2500/lier/rbw20388
https://immovlan.be/en/detail/duplex/for-sale/1000/brussels/vbe35095
https://immovlan.be/en/detail/apartment/for-sale/1210/sint-joost-ten-node/vbe35094
https://immovlan.be/en/detail/residence/for-sale/5300/andenne/vbe35093
https://immovlan.be/en/detail/apartment/for-sale/1050/elsene/vbe35040
https://immovlan.be/en/detail/residence/for-sale/9600/ronse/rbw20337
https://immovlan.be/en/detail/dup

In [5]:

# first looking at the first 10 pages (~200 properties)
links_p2_p10 = []
with requests.Session() as session:
    for i in range(2, 11):
        params_next_page = f"transactiontypes=for-sale&propertytypes=house,apartment&islifeannuity=no&page={i}&noindex=1"

        response = session.get(f"{root}/{end_point}?{params_next_page}", headers={"User-Agent": "max-exercice"})
        link_titles = soup.select('h2 a')
        print(f"Number of links in page {i}: {len(link_titles)}")
        for l in link_titles:
            links_p2_p10.append(l.get("href"))
        



Number of links in page 2: 20
Number of links in page 3: 20
Number of links in page 4: 20
Number of links in page 5: 20
Number of links in page 6: 20
Number of links in page 7: 20
Number of links in page 8: 20
Number of links in page 9: 20
Number of links in page 10: 20


In [6]:
print(len(links_p2_p10), "links : ")
print(*links_p2_p10, sep="\n")


180 links : 
https://immovlan.be/en/detail/residence/for-sale/1150/sint-pieters-woluwe/vbe35189
https://immovlan.be/en/detail/residence/for-sale/6040/jumet/vbe35169
https://immovlan.be/en/detail/residence/for-sale/6183/trazegnies/vbe35167
https://immovlan.be/en/detail/residence/for-sale/5060/moignelee/vbe35133
https://immovlan.be/en/detail/residence/for-sale/5640/mettet/vbe35132
https://immovlan.be/en/detail/duplex/for-sale/5000/namur/vbe35115
https://immovlan.be/en/detail/apartment/for-sale/2390/oostmalle/rbw20430
https://immovlan.be/en/detail/penthouse/for-sale/2500/lier/rbw20388
https://immovlan.be/en/detail/duplex/for-sale/1000/brussels/vbe35095
https://immovlan.be/en/detail/apartment/for-sale/1210/sint-joost-ten-node/vbe35094
https://immovlan.be/en/detail/residence/for-sale/5300/andenne/vbe35093
https://immovlan.be/en/detail/apartment/for-sale/1050/elsene/vbe35040
https://immovlan.be/en/detail/residence/for-sale/9600/ronse/rbw20337
https://immovlan.be/en/detail/duplex/for-sale/713

In [7]:
first_link = link_titles[0].get("href")
response = requests.get(first_link, headers={"User-Agent": "max-exercice"})
print(response.status_code)

200


In [8]:
def finding_purple_properties(soup):
    properties = soup.find_all("li", attrs={"class": "property-highlight"})
    proper_res = []
    if not properties:
        properties = soup.find_all("div", attrs={"class": "property-highlight"})

    bed_already_seen = False
    for p in properties:
        stripped = " ".join([elem.strip().replace("\u202f", " ") for elem in p.text.split("\n") if not all([char == " " for char in elem])])

        if "Bed" in stripped:
            print("found bed")
            if bed_already_seen:
                break
            else:
                bed_already_seen = True
        proper_res.append(stripped)
    return proper_res

In [9]:
soup = BeautifulSoup(response.content, "html")
city = soup.find("span", attrs={"class": "city-line"})
price = soup.find("span", attrs={"class": "detail__header_price_data"})
print(city.text)
print(price.text.strip())
properties = soup.find_all("li", attrs={"class": "property-highlight"})
proper_res = finding_purple_properties(soup)
print(proper_res)



1150 Sint-Pieters-Woluwe
925 000 €
found bed
['5 Bedroom(s)', '200 m²', '2 Bathroom(s)', 'Terrace']


In [10]:
other_link = links_p2_p10[-2]
response = requests.get(other_link, headers={"User-Agent": "max-exercice"})
print(response.status_code)
soup1 = BeautifulSoup(response.content, "html")
city = soup1.find("span", attrs={"class": "city-line"})
price = soup1.find("span", attrs={"class": "detail__header_price_data"})
print(city.text)
print(price.text.strip())

proper_res = finding_purple_properties(soup1)
print(proper_res)


200
6000 Charleroi
From 219 900 €
found bed
['2 Bedroom(s)', '2 Bathroom(s)']


In [11]:
def dic_from_more_info(more_info):
    dic_infos = {}
    h4s = [h.text.replace("\n", "").strip() for h in more_info.find_all("h4")]
    ps = [p.text.replace("\n", "").strip() for p in more_info.find_all("p")]
    for i, title in enumerate(h4s):
        dic_infos[title] = ps[i]
    return dic_infos

more_info = soup1.find("div", attrs={"class": "general-info-wrapper"})
if more_info:
    dic_infos = dic_from_more_info(more_info)
dic_infos

{'Currently leased': 'No',
 'Build Year': '1899',
 'Number of bedrooms': '2',
 'Surface bedroom 1': '20 m²',
 'Surface bedroom 2': '14 m²',
 'Furnished': 'No',
 'Surface of living-room': '40 m²',
 'Kitchen equipment': 'Super equipped',
 'Number of bathrooms': '2',
 'Type of heating': 'Gas',
 'Type of glazing': 'Double glass',
 'Elevator': 'No',
 'Number of facades': '2',
 'Total land surface': '97 m²',
 'Sewer Connection': 'Yes',
 'Gas': 'Yes',
 'Running water': 'Yes',
 'Specific primary energy consumption': '160 kWh/m²/year',
 'Yearly total primary energy consumption': '34068 kWh/year',
 'Validity date EPC/PEB': '17/09/2035',
 'CO2 emission': '30',
 'Flooding Area type': '(information not available)',
 'Demarcated flooding area': '(information not available)'}

In [12]:
df = pd.DataFrame([dic_infos])
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 23 columns):
 #   Column                                   Non-Null Count  Dtype
---  ------                                   --------------  -----
 0   Currently leased                         1 non-null      str  
 1   Build Year                               1 non-null      str  
 2   Number of bedrooms                       1 non-null      str  
 3   Surface bedroom 1                        1 non-null      str  
 4   Surface bedroom 2                        1 non-null      str  
 5   Furnished                                1 non-null      str  
 6   Surface of living-room                   1 non-null      str  
 7   Kitchen equipment                        1 non-null      str  
 8   Number of bathrooms                      1 non-null      str  
 9   Type of heating                          1 non-null      str  
 10  Type of glazing                          1 non-null      str  
 11  Elevator             

In [13]:
soup1

<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8"/>
<meta content="width=device-width, initial-scale=1.0, maximum-scale=1.0" name="viewport"/>
<meta content="app-id=968691127" name="apple-itunes-app"/>
<meta content="app-id=com.czaam" name="google-play-app"/>
<meta content="735734281535384549ac78e4353b4147" name="p:domain_verify"/>
<meta content="Mixed building for sale | 6000 Charleroi | 219 900 € | EPC B | 2 Bedrooms | 2 Bathrooms | Discover all pictures and details." name="description"/>
<meta content="House, Mixed building, for sale, Charleroi, 6000" name="keywords"/>
<meta content="noindex" name="robots"/>
<link href="https://immovlan.be/en/detail/mixed-building/for-sale/6000/charleroi/vbe34871" rel="canonical"/>
<link href="https://immovlan.be/nl/detail/huis-gemengd-gebruik/te-koop/6000/charleroi/vbe34871" hreflang="nl" rel="alternate"/>
<link href="https://immovlan.be/fr/detail/immeuble-mixte/a-vendre/6000/charleroi/vbe34871" hreflang="fr" rel="alternate"/>
<link hre

In [31]:
point_of_interest_title = soup1.find("h2", attrs={"class": "py-3"})
if point_of_interest_title:
    print(True)
    tab1 = soup1.find("div", attrs={"id": "tabs-1"})
    tab1_titles = [elem.text.strip() for elem in tab1.find_all("h3")]
    walking_distances = [elem.text.strip().split()[:2] for elem in tab1.find_all("span", attrs={"title": "Walking"})]
    print(walking_distances)
    walking_distances_m = [wd[0] if wd[1] == 'm' else wd[0] * 1000 for wd in walking_distances]
    print(walking_distances_m)
    education = {key: value for key, value in zip(tab1_titles, walking_distances_m)}
    print(education)

True
[['480', 'm'], ['424', 'm'], ['394', 'm'], ['360', 'm']]
['480', '424', '394', '360']
{'Nurseries': '480', 'Preschools': '424', 'Elementary schools': '394', 'High schools': '360'}


In [52]:
def parsing_tab_of_interest(tab, title):
    #list of sub-titles
    tab_titles = [elem.text.strip() for elem in tab.find_all("h3")]
    #list of walking distances
    distances = [elem.text.strip().split()[:2] for elem in tab.find_all("span", attrs={"title": title})]
    #converting km in m -> list of int
    distances_m = [int(wd[0]) if wd[1] == 'm' else int(float(wd[0]) * 1000) for wd in distances]
    #dictionary with titles and distances
    tab_dico = {key: value for key, value in zip(tab_titles, distances_m)}
    return tab_dico

print(parsing_tab_of_interest(tab1, 'Walking'))

{'Nurseries': 480, 'Preschools': 424, 'Elementary schools': 394, 'High schools': 360}


In [47]:
def parsing_tab_transport(tab):
    tab_dico = {}
    #list of data-rows:
    tab_rows = tab.find_all("div", attrs={"class": "data-row"})
    #list of sub-titles
    tab_titles = [elem.text.strip() for elem in tab.find_all("h3")]
    for i, row in enumerate(tab_rows):
        if i in (0, 4, 5):
            title = "Driving"
        else:
            title = "Walking"

        distance = row.find("span", attrs={"title": title}).text.strip().split()[:2]
        distance = int(distance[0]) if distance[1] == "m" else int(float(distance[0]) * 1000)

        tab_dico[tab_titles[i]] = distance

    return tab_dico

In [57]:
points_of_interest = {}
for i in range(1, 5):

    tab = soup1.find("div", attrs={"id": f"tabs-{i}"})
    if i in (1, 3):
        points_of_interest.update(parsing_tab_of_interest(tab, "Walking"))
    elif i == 2:
        points_of_interest.update(parsing_tab_transport(tab))
    elif i == 4:
        points_of_interest.update(parsing_tab_of_interest(tab, "Driving"))


print(points_of_interest)

{'Nurseries': 480, 'Preschools': 424, 'Elementary schools': 394, 'High schools': 360, 'Motorways': 312, 'Bus': 266, 'Metros': 189, 'Train stations': 1200, 'Airports': 5600, 'Charging stations': 3200, 'Car-sharing system': 641, 'Supermarkets': 15, 'Amusement parks': 9700, 'Museums': 570, 'Religious monuments': 14200, 'Activities': 9800, 'Boat trips': 29300, 'Castles': 25800, 'Golfs': 19200}


In [60]:
for key, value in points_of_interest.items():
    df[key] = value

print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 42 columns):
 #   Column                                   Non-Null Count  Dtype
---  ------                                   --------------  -----
 0   Currently leased                         1 non-null      str  
 1   Build Year                               1 non-null      str  
 2   Number of bedrooms                       1 non-null      str  
 3   Surface bedroom 1                        1 non-null      str  
 4   Surface bedroom 2                        1 non-null      str  
 5   Furnished                                1 non-null      str  
 6   Surface of living-room                   1 non-null      str  
 7   Kitchen equipment                        1 non-null      str  
 8   Number of bathrooms                      1 non-null      str  
 9   Type of heating                          1 non-null      str  
 10  Type of glazing                          1 non-null      str  
 11  Elevator             

Final code for points of interest : only preschool, train station, supermarket

In [61]:
def parsing_preschool(tab):
    #list of sub-titles
    tab_titles = [elem.text.strip() for elem in tab.find_all("h3")]
    if not "Preschools" in tab_titles:
        return None
    #list of walking distances
    distance = [elem.text.strip().split()[:2] for elem in tab.find_all("span", attrs={"title": "Walking"})][tab_titles.index("Preschools")]
    #converting km in m -> list of int
    return int(distance[0]) if distance[1] == 'm' else int(float(distance[0]) * 1000)

In [63]:
def parsing_train_station(tab):
    #list of sub-titles
    tab_titles = [elem.text.strip() for elem in tab.find_all("h3")]
    if "Train stations" not in tab_titles:
        return None

    train_row = tab.find_all("div", attrs={"class": "data-row"})[tab_titles.index("Train stations")]
    
    distance = train_row.find("span", attrs={"title": "Walking"}).text.strip().split()[:2]
    return int(distance[0]) if distance[1] == "m" else int(float(distance[0]) * 1000)


In [62]:
def parsing_supermarket(tab):
    distance = tab.find("span", attrs={"title": "Walking"}).text.strip().split()[:2]
    return int(distance[0]) if distance[1] == 'm' else int(float(distance[0]) * 1000)


In [65]:
points_of_interest = {"Preschool_distance": None, "Train_station_distance": None, "Supermarket_distance": None}
for i in range(1, 4):

    tab = soup1.find("div", attrs={"id": f"tabs-{i}"})
    if tab:
        if i == 1:
            points_of_interest["Preschool_distance"] = parsing_preschool(tab)
        elif i == 2:
            points_of_interest["Train_station_distance"] = parsing_train_station(tab)
        elif i == 3:
            points_of_interest["Supermarket_distance"] = parsing_supermarket(tab)

print(points_of_interest)

{'Preschool_distance': 424, 'Train_station_distance': 1200, 'Supermarket_distance': 15}
